In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import h5py

import pylupnt as pnt

plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"

In [ ]:
path = pnt.get_output_dir("state_estimation") / "example_adaptive.h5"

proc_noises = ["SNC", "ASNC", "ADMC"]
results = {}
with h5py.File(path, "r") as f:
    for proc_noise in proc_noises:
        res = {}
        res["ts"] = np.array(f[f"{proc_noise}/ts"][:]).squeeze()
        res["x_true"] = np.array(f[f"{proc_noise}/x_true"][:])
        res["x_est"] = np.array(f[f"{proc_noise}/x_est"][:])
        res["sigma_est"] = np.array(f[f"{proc_noise}/sigma_est"][:])
        res["N_wait"] = int(f[f"{proc_noise}/N_wait"][()])
        res["N_steps"] = int(f[f"{proc_noise}/N_steps"][()])
        results[proc_noise] = res

N_t = len(results["SNC"]["ts"])

In [ ]:
res = results["ASNC"]
t_start = res["ts"][res["N_wait"] + res["N_steps"]]

plt.figure()
lim = 0.0
for proc_noise in proc_noises:
    res = results[proc_noise]
    err = res["x_true"][:, 0] - res["x_est"][:, 0]
    sig = 3.0 * res["sigma_est"][:, 0]
    plt.plot(res["ts"], err, label=f"{proc_noise} Error")
    plt.fill_between(res["ts"], sig, -sig, alpha=0.2, label=f"{proc_noise} $3\sigma$")
    lim = max(lim, np.max(sig[N_t // 2 :]) * 1.2)

plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.ylim(-lim, lim)
plt.xlim(res["ts"][0], res["ts"][-1])
plt.grid(True)
plt.legend(loc="upper right")
plt.show()

plt.figure()
lim = 0.0
for proc_noise in proc_noises:
    res = results[proc_noise]
    err = res["x_true"][:, 1] - res["x_est"][:, 1]
    sig = 3.0 * res["sigma_est"][:, 1]
    plt.plot(res["ts"], err)
    plt.fill_between(res["ts"], sig, -sig, alpha=0.2)
    lim = max(lim, np.max(sig[N_t // 2 :]) * 1.2)

plt.xlabel("Time [s]")
plt.ylabel("Velocity [m/s]")
plt.ylim(-lim, lim)
plt.xlim(res["ts"][0], res["ts"][-1])
plt.grid(True)
plt.legend()
plt.show()